# Kenya supermarket listings: CRISP-DM

A reproducible study of an advertised-price snapshot. Set `LISTINGS_CSV` or use the relative repository default. APHRC catalogue matching is optional and the source workbook remains local/private.


## 1. Business understanding

Assess snapshot coverage, explore advertised prices, classify product descriptions from the optional APHRC catalogue, and evaluate potential same-product matching only against human labels. This is not a sales, stock, checkout, or price-history dataset.


## 2. Data understanding


In [1]:
from pathlib import Path
import os, re, json
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

ROOT = Path.cwd()
if not (ROOT / 'backend').exists(): ROOT = ROOT.parent
DATA = Path(os.environ.get('LISTINGS_CSV', ROOT / 'backend/data/listings/kenya_supermarket_public_listings.csv'))
df = pd.read_csv(DATA)
print('Dataset:', DATA.name, '| shape:', df.shape)
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().rename('missing').to_frame())
display(df.groupby('supermarket').size().rename('rows').to_frame())
prices = pd.to_numeric(df.price_kes, errors='coerce').dropna()
print('Price quantiles KES:', prices.quantile([0,.25,.5,.75,.9,.95,1]).to_dict())
q1,q3=prices.quantile(.25),prices.quantile(.75); iqr=q3-q1
print('IQR outliers:', int(((prices<q1-1.5*iqr)|(prices>q3+1.5*iqr)).sum()))
display(df.groupby('supermarket').price_kes.agg(['count','median','min','max']))


Dataset: kenya_supermarket_public_listings.csv | shape: (239, 14)


,dtype
supermarket,object
item_name,object
price_kes,float64
pack_size,object
units_sold,float64
currency,object
regular_price_kes,float64
product_url,object
listing_url,object
location_context,object


,missing
supermarket,0
item_name,0
price_kes,0
pack_size,26
units_sold,239
currency,0
regular_price_kes,54
product_url,0
listing_url,0
location_context,198


,rows
supermarket,
Carrefour,25
Naivas,176
Quickmart,38


Price quantiles KES: {0.0: 7.0, 0.25: 120.0, 0.5: 289.0, 0.75: 810.0, 0.9: 2679.000000000007, 0.95: 10544.999999999969, 1.0: 89995.0}
IQR outliers: 30


,count,median,min,max
supermarket,,,,
Carrefour,25,176.0,7.0,612.0
Naivas,176,360.5,45.0,89995.0
Quickmart,38,184.5,47.0,74995.0


## 3. Data preparation

The CSV is the supplied partial snapshot, collected 20 Oct 2026. Missing pack sizes are retained as missing because the source did not provide an unambiguous size. The API parser uses the optional UOM workbook and conservative regular expressions; it only emits unit prices when the size parses unambiguously. Log prices are useful for visualizing a right-skewed distribution, not for replacing KES values.


In [2]:
df['price_kes'] = pd.to_numeric(df.price_kes, errors='coerce')
df['log_price'] = np.log1p(df.price_kes)
regular = pd.to_numeric(df.regular_price_kes, errors='coerce')
discount = (regular-df.price_kes)/regular
print('Rows with regular price:', int(regular.notna().sum()))
print('Mean displayed discount among rows with regular price:', round(float(discount.dropna().mean()),4))
print('Offer skew: descriptive snapshot statistic; promotions and homepage selection may bias it.')


Rows with regular price: 185
Mean displayed discount among rows with regular price: 0.2492
Offer skew: descriptive snapshot statistic; promotions and homepage selection may bias it.


## 4. Modelling

### 4a. Category classification (APHRC optional)


In [3]:
workbook = ROOT / 'backend/data/listings/private/APHRC_supermarket_item_catalogue_and_dictionary.xlsx'
if not workbook.exists():
    print('Skipped category training: optional APHRC workbook not present.')
else:
    catalogue = pd.read_excel(workbook, sheet_name='unique_items')
    target = 'food_group'
    training = catalogue.dropna(subset=['description', target]).copy()
    training['group'] = training.description.astype(str).str.lower().str.replace(r'\W+', ' ', regex=True).str.strip()
    splitter = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42)
    train_i, test_i = next(splitter.split(training.description, training[target], groups=training.group))
    Xtr,Xte=training.description.iloc[train_i],training.description.iloc[test_i]
    ytr,yte=training[target].iloc[train_i],training[target].iloc[test_i]
    candidates={'Dummy majority':DummyClassifier(strategy='most_frequent'), 'MultinomialNB':make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2),MultinomialNB()), 'DecisionTree':make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2),DecisionTreeClassifier(max_depth=30, random_state=42))}
    for name,model in candidates.items():
        model.fit(Xtr,ytr); pred=model.predict(Xte)
        print(name,'accuracy',round(accuracy_score(yte,pred),4))
        print(classification_report(yte,pred,zero_division=0))
        display(pd.DataFrame(confusion_matrix(yte,pred,labels=model.classes_),index=model.classes_,columns=model.classes_))
    label_path=ROOT/'backend/data/labels/category_sample_to_label.csv'
    labels=pd.read_csv(label_path) if label_path.exists() else pd.DataFrame()
    filled=labels[labels.label.notna() & labels.label.astype(str).str.strip().ne('')] if not labels.empty else labels
    if len(filled):
        print('CSV human-labelled sample rows:',len(filled),'(evaluate after aligning fitted model/domain labels)')
    else: print('labels not filled yet: edit backend/data/labels/category_sample_to_label.csv')
    print('Domain shift: APHRC till-style catalogue descriptions differ from retailer listing wording.')


Dummy majority accuracy 0.1275
                                          precision    recall  f1-score   support

                              Baby Foods       0.00      0.00      0.00         7
                   Beverages - Sweetened       0.00      0.00      0.00       143
                 Beverages - Unsweetened       0.00      0.00      0.00        78
              Cereals and Grain products       0.00      0.00      0.00       134
            Condiments, Spices and Herbs       0.13      1.00      0.23       154
 Confectionery (Candy, Gums, Chocolates)       0.00      0.00      0.00        62
                       Fish and Sea Food       0.00      0.00      0.00         3
Food Colour/Flavour and Baking Additives       0.00      0.00      0.00        15
                   Fruits and Vegetables       0.00      0.00      0.00        42
                      Legumes and Pulses       0.00      0.00      0.00        14
                  Meat, Poultry and Eggs       0.00      0.00     

,Baby Foods,Beverages - Sweetened,Beverages - Unsweetened,Cereals and Grain products,"Condiments, Spices and Herbs","Confectionery (Candy, Gums, Chocolates)",Fish and Sea Food,Food Colour/Flavour and Baking Additives,Fruits and Vegetables,Legumes and Pulses,"Meat, Poultry and Eggs",Milk and Dairy Products,Mixed Dishes and Fast Foods,Nuts and Seeds,Oils and Fat,Savoury snacks,Starchy Roots and Tubers,Sugar and Sweetened Products,Sweet Snacks/Sweet Baked Foods
Baby Foods,0,0,0,0,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Beverages - Sweetened,0,0,0,0,143,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Beverages - Unsweetened,0,0,0,0,78,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Cereals and Grain products,0,0,0,0,134,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"Condiments, Spices and Herbs",0,0,0,0,154,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"Confectionery (Candy, Gums, Chocolates)",0,0,0,0,62,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Fish and Sea Food,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Food Colour/Flavour and Baking Additives,0,0,0,0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Fruits and Vegetables,0,0,0,0,42,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Legumes and Pulses,0,0,0,0,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0


MultinomialNB accuracy 0.8849
                                          precision    recall  f1-score   support

                              Baby Foods       0.00      0.00      0.00         7
                   Beverages - Sweetened       0.88      0.95      0.92       143
                 Beverages - Unsweetened       0.94      0.82      0.88        78
              Cereals and Grain products       0.81      0.97      0.88       134
            Condiments, Spices and Herbs       0.74      0.99      0.84       154
 Confectionery (Candy, Gums, Chocolates)       0.98      0.87      0.92        62
                       Fish and Sea Food       0.00      0.00      0.00         3
Food Colour/Flavour and Baking Additives       1.00      0.40      0.57        15
                   Fruits and Vegetables       0.96      0.57      0.72        42
                      Legumes and Pulses       1.00      0.43      0.60        14
                  Meat, Poultry and Eggs       1.00      0.89      

,Baby Foods,Beverages - Sweetened,Beverages - Unsweetened,Cereals and Grain products,"Condiments, Spices and Herbs","Confectionery (Candy, Gums, Chocolates)",Fish and Sea Food,Food Colour/Flavour and Baking Additives,Fruits and Vegetables,Legumes and Pulses,"Meat, Poultry and Eggs",Milk and Dairy Products,Mixed Dishes and Fast Foods,Nuts and Seeds,Oils and Fat,Savoury snacks,Starchy Roots and Tubers,Sugar and Sweetened Products,Sweet Snacks/Sweet Baked Foods
Baby Foods,0,0,0,6,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Beverages - Sweetened,0,136,4,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Beverages - Unsweetened,0,13,64,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
Cereals and Grain products,0,0,0,130,1,0,0,0,0,0,0,0,0,0,0,0,0,0,3
"Condiments, Spices and Herbs",0,0,0,1,152,0,0,0,0,0,0,1,0,0,0,0,0,0,0
"Confectionery (Candy, Gums, Chocolates)",0,0,0,0,2,54,0,0,0,0,0,1,0,0,0,0,0,0,5
Fish and Sea Food,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,1
Food Colour/Flavour and Baking Additives,0,1,0,0,7,0,0,6,0,0,0,1,0,0,0,0,0,0,0
Fruits and Vegetables,0,1,0,1,16,0,0,0,24,0,0,0,0,0,0,0,0,0,0
Legumes and Pulses,0,0,0,6,2,0,0,0,0,6,0,0,0,0,0,0,0,0,0


DecisionTree accuracy 0.6026
                                          precision    recall  f1-score   support

                              Baby Foods       0.00      0.00      0.00         7
                   Beverages - Sweetened       0.99      0.62      0.76       143
                 Beverages - Unsweetened       0.90      0.56      0.69        78
              Cereals and Grain products       0.22      0.99      0.37       134
            Condiments, Spices and Herbs       0.91      0.47      0.62       154
 Confectionery (Candy, Gums, Chocolates)       1.00      0.47      0.64        62
                       Fish and Sea Food       0.00      0.00      0.00         3
Food Colour/Flavour and Baking Additives       0.50      0.07      0.12        15
                   Fruits and Vegetables       0.89      0.38      0.53        42
                      Legumes and Pulses       0.00      0.00      0.00        14
                  Meat, Poultry and Eggs       0.96      0.51      0

,Baby Foods,Beverages - Sweetened,Beverages - Unsweetened,Cereals and Grain products,"Condiments, Spices and Herbs","Confectionery (Candy, Gums, Chocolates)",Fish and Sea Food,Food Colour/Flavour and Baking Additives,Fruits and Vegetables,Legumes and Pulses,"Meat, Poultry and Eggs",Milk and Dairy Products,Mixed Dishes and Fast Foods,Nuts and Seeds,Oils and Fat,Savoury snacks,Starchy Roots and Tubers,Sugar and Sweetened Products,Sweet Snacks/Sweet Baked Foods
Baby Foods,0,0,0,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Beverages - Sweetened,0,88,5,48,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0
Beverages - Unsweetened,0,0,44,31,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Cereals and Grain products,0,0,0,133,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
"Condiments, Spices and Herbs",0,0,0,80,72,0,0,0,1,0,1,0,0,0,0,0,0,0,0
"Confectionery (Candy, Gums, Chocolates)",0,1,0,30,0,29,0,0,0,0,0,2,0,0,0,0,0,0,0
Fish and Sea Food,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Food Colour/Flavour and Baking Additives,0,0,0,14,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
Fruits and Vegetables,0,0,0,25,1,0,0,0,16,0,0,0,0,0,0,0,0,0,0
Legumes and Pulses,0,0,0,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


labels not filled yet: edit backend/data/labels/category_sample_to_label.csv
Domain shift: APHRC till-style catalogue descriptions differ from retailer listing wording.


### 4b. Same-product matching

Candidate generation blocks on normalized pack size and shared name tokens. Pair similarity combines TF-IDF cosine and token-set overlap. Threshold selection and held-out metrics are only produced when human labels are entered.


In [4]:
pairs_path=ROOT/'backend/data/labels/match_pairs_to_label.csv'
if pairs_path.exists():
    pairs=pd.read_csv(pairs_path)
    labelled=pairs[pairs.label.notna() & pairs.label.astype(str).str.strip().ne('')]
else: labelled=pd.DataFrame()
if len(labelled)<4:
    print('labels not filled yet: edit backend/data/labels/match_pairs_to_label.csv')
else:
    from sklearn.model_selection import train_test_split
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    names=(labelled.name_a.fillna('')+' '+labelled.name_b.fillna('')).tolist()
    vec=TfidfVectorizer().fit_transform(names)
    cosine=np.diag(cosine_similarity(vec[:len(names)//2],vec[len(names)//2:])) if len(names)%2==0 else np.zeros(len(names))
    # With the supplied pair records, token-set ratio is an interpretable companion signal.
    def token_ratio(a,b):
        A=set(re.findall(r'[a-z0-9]+',str(a).lower())); B=set(re.findall(r'[a-z0-9]+',str(b).lower())); return 2*len(A&B)/max(1,len(A)+len(B))
    score=np.array([.5*token_ratio(a,b)+.5*cosine_similarity(TfidfVectorizer().fit_transform([str(a),str(b)]))[0,1] for a,b in zip(labelled.name_a,labelled.name_b)])
    y=labelled.label.astype(int).to_numpy(); ix=np.arange(len(y)); tr,te=train_test_split(ix,test_size=.3,random_state=42,stratify=y)
    thresholds=np.linspace(0,1,101); best=max(thresholds,key=lambda t: precision_recall_fscore_support(y[tr],score[tr]>=t,average='binary',zero_division=0)[2])
    pred=score[te]>=best
    print('Threshold:',round(float(best),3),'held-out rows:',len(te),'held-out positives:',int(y[te].sum()))
    print('Confusion matrix [0,1]:\n',confusion_matrix(y[te],pred,labels=[0,1]))
    print('Precision/recall/F1:',precision_recall_fscore_support(y[te],pred,average='binary',zero_division=0)[:3])


labels not filled yet: edit backend/data/labels/match_pairs_to_label.csv


### 4c. Unit prices, store summaries, and optional clustering

Unit-price summaries are limited to parsed comparable units. Discount comparisons are descriptive only and are affected by offer-skew.


In [5]:
from price_check_api.listings import get_listings
items=get_listings()
unit=pd.DataFrame([{'store':x.supermarket,'category':x.category,'unit':x.unit,'unit_price':x.unit_price} for x in items if x.unit_price is not None])
if len(unit):
    display(unit.groupby(['category','unit','store']).unit_price.agg(['count','median','min','max']))
    q1=unit.groupby(['category','unit']).unit_price.transform(lambda s:s.quantile(.25)); q3=unit.groupby(['category','unit']).unit_price.transform(lambda s:s.quantile(.75)); iqr=q3-q1
    print('Within category/unit IQR outliers:',int(((unit.unit_price<q1-1.5*iqr)|(unit.unit_price>q3+1.5*iqr)).sum()))
else: print('No unambiguous unit prices to summarize.')
print('No price-history or causal inference is possible from one advertised snapshot.')


count      median        min         max
category      unit store                                              
bakery        kg   Naivas         3   449.00000   122.5000    495.0000
beverages     kg   Carrefour      1  6666.66670  6666.6667   6666.6667
                   Quickmart      1   495.00000   495.0000    495.0000
              l    Carrefour      3    88.00000    88.0000    154.0000
                   Naivas         4  1264.54885   365.6250   2498.1203
dairy         kg   Naivas         6   505.50000   475.0000    550.0000
                   Quickmart      3   424.00000   424.0000    495.0000
              l    Carrefour      3   184.00000   130.0000    880.0000
                   Quickmart      3   188.00000   136.0000    280.0000
              pcs  Carrefour      3    20.40000    16.8000     21.9333
household     kg   Naivas         5   800.00000   352.0000   1422.8571
              l    Naivas         2   570.00000   570.0000    570.0000
                   Quickmart      4   347.50000   325.0000    370.0000
non_grocery   kg   Naivas         1  6699.50000  6699.5000   6699.5000
              l    Naivas         7  1332.00000   148.4798   3465.3333
              pcs  Naivas         1   699.50000   699.5000    699.5000
pantry        kg   Carrefour      9   272.50000    45.0000   2020.8333
                   Naivas        61   437.50000    32.4000   2444.4444
                   Quickmart      5   495.00000   495.0000   1042.1053
              l    Carrefour      6   135.00000    88.0000    202.0000
                   Naivas        31   900.00000    74.5000  36600.0000
                   Quickmart      4   627.76925   184.0000  25666.6667
              pcs  Naivas         6    24.96095     0.9500     49.8333
                   Quickmart      1     0.49580     0.4958      0.4958
personal_care l    Naivas         5  1700.00000  1462.5000   3833.3333
produce       kg   Naivas         8  1117.50000   282.8571   3552.6316
              l    Naivas         2   928.90000   259.8000   1598.0000

Within category/unit IQR outliers: 18
No price-history or causal inference is possible from one advertised snapshot.


## 5. Evaluation

Category metrics are grouped-holdout performance on APHRC text and are not retailer accuracy. Matching is reported only for user-provided labels; a tiny pair sample has high uncertainty. There is no invented label performance.


## 6. Deployment

The FastAPI listing endpoints expose snapshot date, source metadata, and caveats. Production deployment would need permitted data access, repeat collection, refreshed labels, drift monitoring, privacy review, and verification of stock and checkout prices with retailers. This notebook makes no deployment claim.
